# Finance Advisor (No Memory)

A personal finance advisor agent built with the OpenAI Agents SDK.

- Tracks spending by category and gives budget advice based on what's in **the current message only**.
- **No memory / session is attached.** Each call to `Runner.run` is independent — the agent has no access to any earlier query in this notebook. This is intentional, to demonstrate stateless behavior as a baseline before adding a memory-enabled version.
- Uses `await Runner.run(...)` rather than `Runner.run_sync(...)`: Jupyter/ipykernel already runs its own asyncio event loop, and `run_sync` raises `RuntimeError: cannot be called when an event loop is already running` in that context.
- Model: `gpt-5-mini`

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — check your .env file."
print("Environment loaded. OPENAI_API_KEY is set.")

Environment loaded. OPENAI_API_KEY is set.


## Agent Definition

The `compute_budget_status` tool does the arithmetic (remaining budget, percent used) whenever the agent has both a weekly budget figure and a total-spent figure available *within the current query*. It holds no state of its own between calls.

In [2]:
from typing import Annotated

from agents import Agent, Runner, function_tool


@function_tool
def compute_budget_status(
    weekly_budget: Annotated[float, "The user's weekly budget in dollars"],
    total_spent: Annotated[float, "Total amount spent so far this week, in dollars"],
) -> dict:
    """Compute remaining budget and percent of budget used for the current week."""
    remaining = round(weekly_budget - total_spent, 2)
    percent_used = round((total_spent / weekly_budget) * 100, 1) if weekly_budget else 0.0
    return {
        "weekly_budget": weekly_budget,
        "total_spent": total_spent,
        "remaining": remaining,
        "percent_used": percent_used,
        "over_budget": remaining < 0,
    }


finance_advisor = Agent(
    name="Finance Advisor",
    instructions=(
        "You are Finance Advisor, a personal finance assistant. "
        "For every user message, extract only the spending amounts and categories mentioned "
        "in THAT message (e.g. groceries, transport/Uber, restaurants/dining, entertainment, "
        "utilities, shopping, other) and summarize spending by category. "
        "If the user gives a weekly budget and you also have a total-spent figure from the "
        "current message, call compute_budget_status to calculate the remaining budget and "
        "give specific, practical advice on where to cut back next week, prioritizing "
        "discretionary categories (restaurants, entertainment, shopping) over essentials "
        "(groceries, utilities). "
        "IMPORTANT: You do not have memory of previous messages or sessions — each query is "
        "handled completely independently. If the user refers to spending or context from "
        "'earlier' or 'so far' that is not present in the current message, explicitly say you "
        "don't have access to prior conversation history and ask them to restate the figures."
    ),
    model="gpt-5-mini",
    tools=[compute_budget_status],
)

## Test Query 1

> I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.

In [3]:
query_1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."

result_1 = await Runner.run(finance_advisor, query_1)
print(result_1.final_output)

Summary of spending mentioned in this message:
- Groceries: $120
- Transport / Uber: $40
- Restaurants / Dining: $60
Total this week (from this message): $220

Practical advice — where to cut next week (prioritizing discretionary categories):
- Restaurants / Dining ($60): Highest-discretion area. Aim to cut this first.
  - Target: reduce by $20–$30 (e.g., dine out once instead of twice, or choose cheaper options) to save $20–$30.
  - Tactics: plan one “treat” meal, cook one restaurant-style meal at home, use lunch leftovers, and avoid impulse orders.
- Transport / Uber ($40): Moderately discretionary.
  - Target: reduce by $10–$20 (use public transit, walk, bike, or combine trips) to save $10–$20.
  - Tactics: combine errands into one trip, schedule rides during off-peak or use promo codes, consider a short public-transit route for regular trips.
- Groceries ($120): Essential but can be optimized.
  - Target: trim $10–$25 with meal planning and smarter shopping.
  - Tactics: make a sho

## Test Query 2

> My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?

This is run as a brand-new, independent call (no session passed to `Runner.run`), so the agent has **no memory** of Query 1. Expect it to say it doesn't have the earlier spending details rather than recalling the $120/$40/$60 figures.

In [4]:
query_2 = "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?"

result_2 = await Runner.run(finance_advisor, query_2)
print(result_2.final_output)

Summary of amounts in this message
- Weekly budget: $250
- No spending amounts or categories were given in this message.

I don't have access to prior conversation history, so I can't see the spending breakdown you referred to ("everything I told you so far"). Please restate this week's spending (or total spent so far) broken down by category so I can run the remaining-budget calculation and give tailored cuts.

If helpful, paste numbers using this template:
- Groceries: $____
- Transport / Uber: $____
- Restaurants / Dining: $____
- Entertainment: $____
- Utilities: $____
- Shopping: $____
- Other: $____
- Total spent so far this week: $____

Quick, actionable guidance you can apply immediately (prioritizing discretionary cuts)
- Cap dining out: cook more meals and set a hard dining budget (e.g., $20–30/week). Cooking one extra meal per day can cut restaurant spend dramatically.
- Entertainment: choose free/low-cost options (streaming instead of tickets, free events, walks). Aim to cu

## Task 3: Adding Memory with `SQLiteSession`

This section adds persistent conversation memory using `SQLiteSession`, so the agent can recall spending mentioned in earlier messages within the same session — unlike the Task 1/2 section above, which is intentionally stateless.

Model: `gpt-5-mini` (re-verified before writing this section — the dated snapshot `gpt-5-mini-2025-08-07` is scheduled to retire December 10, 2026, but the `gpt-5-mini` alias itself remains active as of July 2026, so it's used here exactly as specified, unchanged from Task 1/2).

In [5]:
from agents import SQLiteSession

# Fresh session for this demo — a new SQLiteSession id starts with empty history.
memory_session = SQLiteSession("finance_advisor_demo")

finance_advisor_with_memory = Agent(
    name="Finance Advisor",
    instructions=(
        "You are Finance Advisor, a personal finance assistant with memory of this conversation. "
        "Track every spending amount and category the user has mentioned across the ENTIRE "
        "conversation history available to you (not just the latest message) and summarize "
        "spending by category. "
        "If you have both a weekly budget and total spending figures from anywhere in this "
        "session, call compute_budget_status to calculate the remaining budget and give "
        "specific, practical advice on where to cut back next week, prioritizing discretionary "
        "categories (restaurants, entertainment, shopping) over essentials (groceries, "
        "utilities). "
        "Use the conversation history available to you via session memory — if the user refers "
        "to spending 'so far' or 'earlier', recall it from prior turns in this session instead "
        "of asking them to repeat it."
    ),
    model="gpt-5-mini",
    tools=[compute_budget_status],
)

### Test Query 1 (with memory)

> I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.

In [6]:
memory_query_1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."

memory_result_1 = await Runner.run(finance_advisor_with_memory, memory_query_1, session=memory_session)
print(memory_result_1.final_output)

Got it — I’ve recorded these for this week. Summary of what you’ve spent so far this week:
- Groceries: $120 (essential)
- Uber: $40 (discretionary / transport)
- Restaurants: $60 (discretionary)

Total this week: $220

Quick, prioritized suggestions to cut next week (focus on discretionary first)

1. Restaurants (highest-priority cut)
   - Aim to cut by 30–50%: e.g., reduce $60 → $30 (save $30).
   - Practical steps: plan 2–3 cook-at-home meals, batch-cook lunches for work, limit takeout to one treat meal, use leftovers creatively.

2. Uber / rideshares
   - Aim to cut by 25–50%: e.g., $40 → $20 (save $20).
   - Practical steps: combine errands into one trip, use public transit/walking/biking if safe, schedule fewer short rides, use pooled/shared rides or promo codes.

3. Groceries (least immediate priority)
   - Try modest savings ($5–$20): shop a list, buy store brands, use sales/coupons, buy staples in bulk, avoid impulse buys.
   - Even a small 10% reduction = $12 saved.

Example 

### Test Query 2 (with memory)

> My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?

The same `memory_session` is passed again, so `SQLiteSession` automatically loads Query 1's history before this call. Expect the agent to reference the $120/$40/$60 figures from Query 1 this time, unlike the no-memory version above.

In [7]:
memory_query_2 = "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?"

memory_result_2 = await Runner.run(finance_advisor_with_memory, memory_query_2, session=memory_session)
print(memory_result_2.final_output)

Summary (what you’ve told me so far)
- Groceries: $120 (essential)
- Uber: $40 (discretionary / transport)
- Restaurants: $60 (discretionary)
- Total spent this week: $220
- Weekly budget: $250
- Remaining this week: $30 (you’ve used 88% of your budget)

Where to cut next week (prioritized, practical steps)

1) Restaurants — highest-priority cut (bigest discretionary)
- Why: discretionary, easy to reduce quickly.
- Target reductions and impact:
  - Cut to $30 (50%): saves $30 → new remaining = $60.
  - Cut to $15 (75%): saves $45 → new remaining = $75.
- Practical steps: plan 3–5 cook-at-home meals, batch-cook lunches, limit takeout to one treat meal, use simple recipes (stir-fries, pasta, rice-bowls), freeze leftovers, set a restaurant allowance on your phone.

2) Uber / rideshares — next priority
- Why: discretionary transport; alternatives often available.
- Target reductions and impact:
  - Cut by 25% → $30 (save $10) → new remaining = $40.
  - Cut by 50% → $20 (save $20) → new rem

## Side-by-Side Comparison: No Memory vs. With Memory

Both sections were asked the identical Query 2: *"My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?"*

In [8]:
print("=" * 70)
print("WITHOUT MEMORY (Task 1/2, no session)")
print("=" * 70)
print(result_2.final_output)
print()
print("=" * 70)
print("WITH MEMORY (Task 3, SQLiteSession)")
print("=" * 70)
print(memory_result_2.final_output)

WITHOUT MEMORY (Task 1/2, no session)
Summary of amounts in this message
- Weekly budget: $250
- No spending amounts or categories were given in this message.

I don't have access to prior conversation history, so I can't see the spending breakdown you referred to ("everything I told you so far"). Please restate this week's spending (or total spent so far) broken down by category so I can run the remaining-budget calculation and give tailored cuts.

If helpful, paste numbers using this template:
- Groceries: $____
- Transport / Uber: $____
- Restaurants / Dining: $____
- Entertainment: $____
- Utilities: $____
- Shopping: $____
- Other: $____
- Total spent so far this week: $____

Quick, actionable guidance you can apply immediately (prioritizing discretionary cuts)
- Cap dining out: cook more meals and set a hard dining budget (e.g., $20–30/week). Cooking one extra meal per day can cut restaurant spend dramatically.
- Entertainment: choose free/low-cost options (streaming instead of t